# Introduction to primitives

In [24]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(name= "General_Usage")
# service.saved_accounts()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

print(backend.name)

# for using fractional gates
# fractional_gate_backend = service.least_busy(use_fractional_gates=True)

ibm_fez


## 1. Estimator primitive

- ####  Create a circuit and an observable

In [2]:
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp

observable = SparsePauliOp("Z" * 10) # expectation value of ZZZ...Z (on 10 qubits)
circuit = qaoa_ansatz(observable, reps=2)
# the circuit is parametrized, so we will define the parameter values for execution
param_values = [0.1, 0.2, 0.3, 0.4]

print(f">>> Observable: {observable.paulis}")

>>> Observable: ['ZZZZZZZZZZ']


- #### Transformed to ISA

The circuit and observable need to be transformed to only use instructions supported by the QPU (referred to as instruction set architecture (ISA) circuits). We'll use the transpiler to do this.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 136), ('sx', 50), ('cz', 36)])


- #### Initialize Qiskit Runtime Estimator

When you initialize the Estimator, use the mode parameter to specify the mode you want it to run in. Possible values are batch, session, or backend objects for batch, session, and job execution mode, respectively. 

In [4]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)

- #### Invoke the Estimator and get results

Next, invoke the `run()` method to calculate expectation values for the input circuits and observables. The circuit, observable, and optional parameter value sets are input as primitive unified bloc (PUB) tuples.

In [12]:
job = estimator.run([(isa_circuit, isa_observable, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d7gts8q2khts739p0p10
>>> Job Status: QUEUED


In [13]:
result = job.result()
print(f">>> {result}")
print(f"  > Expectation value: {result[0].data.evs}")
print(f"  > Metadata: {result[0].metadata}")

>>> PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})
  > Expectation value: -0.1965556831228473
  > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


## 2. Sampler primitive

The steps of executing using a sampler is similar to the steps used for an Estimator. Only main difference is there is no observable here, hence no invoking `apply_layout` which transforms observable to ISA suitable operations.

In [24]:
import numpy as np
from qiskit.circuit.library import efficient_su2

circuit = efficient_su2(75, entanglement="linear")
circuit.measure_all()
# The circuit is parametrized, so we will define the parameter values for execution
param_values = np.random.rand(circuit.num_parameters)

In [25]:
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 1788), ('sx', 1041), ('cz', 222), ('measure', 75), ('barrier', 1)])


In [26]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

In [20]:
job = sampler.run([(isa_circuit, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d7gv98nb91ec73au2ck0
>>> Job Status: QUEUED


In [ ]:
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]
print(

    f"First ten results for the 'meas' output register: {pub_result.data.meas.get_bitstrings()[:10]}" # this is a list of bitstrings
)
print(f"Bitstring Counts for the first ten bitstrings: {sorted(pub_result.data.meas.get_counts().items(), key=lambda kv: kv[1], reverse=True)[:5]}")
 # pub_result.data.meas.get_counts() is a dictionary of bitstring:count

print(f"Integer Counts for the first ten bitstrings: {sorted(pub_result.data.meas.get_int_counts().items(), key=lambda kv: kv[1], reverse=True)[:5]}")


First ten results for the 'meas' output register: ['101111001100000000000011001110001110101000110111011000011000001100111110010', '100000000011100010001001001101010110101101001010001000100001010000110110110', '101100001100110010011010011001110100110001000000001010101010101001111000000', '110111001101101100000110011000110111000000001001101101000000110110101010100', '101011001001100110011011000000001011010101011111101010010101010100000101100', '101010001111111011110100000010001111000110000111000010100110110100001010101', '000001010000111111110000101010101111100110011001010011010101010101010101010', '000100111000000011000100011010111010101110001010101001010101011110011010000', '101010100110111010001001001100101101110101000000001010010001111110001110000', '110110101011100110011001101111111001110000101001001110000001010101001000100']
Counts for the first ten bitstrings: [('101111001100000000000011001110001110101000110111011000011000001100111110010', 1), ('10000000001110001000100100110101011

### Shot option for Sampler

In [ ]:
# # Instantiate with a default
sampler_with_options = Sampler(mode=backend, options={"default_shots": 1000})

# Run a job WITH shots specified in run() - this should override the default
job2 = sampler_with_options.run([(isa_circuit, param_values)], shots=2000)
result2 = job2.result()
print(f"Run 2 used the overridden shots: {result2[0].data.meas.num_shots}")

Run 2 used the overridden shots: 2000


#### Note: Sampler and Estimator are IBM Quantum provider (qiskit runtime) specific primitives and not generic to any backend provider.

## Backend primitives

Backend primitives are generic implementations that can be used with an arbitrary backend object, as long as it implements the `Backend` interface.

- The `Sampler` primitive can be run with any provider by using `qiskit.primitives.BackendSamplerV2`.
- The `Estimator` primitive can be run with any provider by using `qiskit.primitives.BackendEstimatorV2`.

``` python 
from qiskit.primitives import BackendEstimatorV2,  BackendSamplerV2
from <some_qiskit_provider> import QiskitProvider

provider = QiskitProvider()
backend = provider.get_backend('backend_name')
estimator = BackendEstimatorV2(backend)
sampler = BackendSamplerV2(backend)


In [ ]:
# we demonstrate using the AerSimulator provider
from qiskit_aer import AerSimulator
backend = AerSimulator()

print(f"Using local backend: {backend.name}")

from qiskit.primitives import BackendEstimatorV2 as Estimator
estimator = Estimator(backend=backend) # notice its backend and not mode 


num_qubits = 5
entanglement = [(0, 1), (1, 2), (2, 3), (3, 4)]
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 0.5) for i, j in entanglement],
    num_qubits=num_qubits,
)
circuit = qaoa_ansatz(observable, reps=2)
# The parameter values now need to match the smaller circuit's parameter count
param_values = np.random.rand(circuit.num_parameters)

print(f">>> Circuit has {circuit.num_parameters} parameters.")
print(f">>> Observable Paulis: {observable.paulis}")

# The run call takes a list of Primitives Unified Blocs (PUBs).
# Each PUB is a tuple of (circuit, observable, [parameter_values]).
job = estimator.run([(circuit, observable, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")
result = job.result()
print(f"\n>>> Full Result Object: {result}")
# The result for the first PUB is at index 0
pub_result = result[0]
print(f"\n  > Expectation value (EV): {pub_result.data.evs}")
print(f"  > Standard deviation: {pub_result.data.stds}")
print(f"  > Metadata: {pub_result.metadata}")

Using local backend: aer_simulator
>>> Circuit has 4 parameters.
>>> Observable Paulis: ['IIIZZ', 'IIZZI', 'IZZII', 'ZZIII']
>>> Job ID: 6184a0c6-dd4b-4c51-a0e6-83bfbc4235c2
>>> Job Status: JobStatus.RUNNING

>>> Full Result Object: PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.float64(0.031077357111046918)), metadata={'target_precision': 0.015625, 'shots': 4096, 'circuit_metadata': {}})], metadata={'version': 2})

  > Expectation value (EV): -0.192626953125
  > Standard deviation: 0.031077357111046918
  > Metadata: {'target_precision': 0.015625, 'shots': 4096, 'circuit_metadata': {}}


## Exact simulation with Qiskit SDK primitives

The reference primitives in the Qiskit SDK perform local statevector simulations. These simulations do not support modeling device noise, but are useful for quickly prototyping algorithms before looking into more advanced simulation techniques (using Qiskit Aer) or running on real devices (Qiskit Runtime primitives).


## Use the reference Estimator

The reference implementation of `EstimatorV2` in `qiskit.primitives` that runs on a local statevector simulators is the `StatevectorEstimator` class. It can take circuits, observables, and parameters as inputs and returns the locally computed expectation values.

The following code prepares the inputs that will be used in the examples that follow. The expected input type for the observables is qiskit.quantum_info.SparsePauliOp. Note that the circuit in the example is parametrized, but you can also run the Estimator on non-parametrized circuits.

**Any circuit passed to an Estimator must not include any measurements.**

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

# observable(s) whose expected values you want to compute
observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

# Generate a pass manager without providing a backend, the below codes are optional here
# The reference primitives still accept abstract instructions, as they rely on local statevector simulations, 
# but transpiling the circuit could be beneficial in terms of circuit optimization.

# from qiskit.transpiler import generate_preset_pass_manager
# pm = generate_preset_pass_manager(optimization_level=1) 
# isa_circuit = pm.run(circuit)
# isa_observable = observable.apply_layout(isa_circuit.layout)


from qiskit.primitives import StatevectorEstimator
estimator = StatevectorEstimator()

job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()

#The primitives result outputs an array of PubResult objects, where each item of the array is a PubResult object 
# that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

print(f" > Expectation value: {result[0].data.evs}") # the expectation value for the first (and only) circuit-observable combination in the PUB
print(f" > Metadata: {result[0].metadata}")


 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### Set Estimator run options

By default, the reference Estimator performs an exact statevector calculation based on the `quantum_info.Statevector` class. However, this can be modified to introduce the effect of the sampling overhead (also known as "shot noise").

Estimator accepts a precision argument that expresses the error bars that the primitive implementation should target for expectation values estimates. This is the sampling overhead and is defined exclusively in the .run() method. This lets you fine-tune the option all the way to the PUB level.

In [ ]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the `StatevectorSampler` class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

**Any quantum circuit passed to a Sampler must include measurements.**

In [31]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

# optional, just like before, the reference primitives still accept abstract instructions, as they rely on local statevector simulations,
# Generate a pass manager without providing a backend
# from qiskit.transpiler import generate_preset_pass_manager

# pm = generate_preset_pass_manager(optimization_level=1)
# isa_circuit = pm.run(circuit)

from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")


#Primitives accept multiple PUBs as inputs, and each PUB gets its own result. 
# Therefore, you can run different circuits with various parameter/observable combinations, and retrieve the PUB results:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]


 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


**The name of the classical register defaults to "meas". This name will be used later to access the measurement bitstrings.**

In [32]:

# Access result data for PUB 1
data_pub = pub_result_1.data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts for first circuit is: {counts}")

# Access result data for PUB 2
data_pub = pub_result_2.data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts for second circuit is: {counts}")

The number of bitstrings is: 1024
The counts for first circuit is: {'00': 492, '11': 532}
The number of bitstrings is: 1024
The counts for second circuit is: {'11': 529, '00': 495}


#### Change run options

By default, the reference Sampler performs an exact statevector calculation based on the quantum_info.Statevector class. However, this can be modified to introduce the effect of the sampling overhead (also known as "shot noise"). To help manage this overhead, the Sampler interface accepts a shots argument that can be defined at the PUB level.

In [ ]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=5000)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 2000), (isa_circuit2, None, 6000)])

## Error mitigation and suppression techniques

### Dynamical decoupling

Quantum circuits on IBM hardware are implemented as carefully timed microwave pulses. During execution, some qubits inevitably sit idle while others are being operated on. These idle periods are risky: unwanted interactions with the environment or neighboring qubits can introduce errors.

Dynamical decoupling (DD) tackles this by inserting additional pulse sequences on idle qubits. These sequences are designed to act as an overall identity (i.e., they don’t change the logical state), but they actively cancel out error accumulation by averaging unwanted interactions over time.

However, DD is not universally helpful. It works best when qubits have idle gaps. 
If the circuit is already densely packed, adding pulses can backfire—more operations → more opportunities for imperfections.

#### Coherent vs. Incoherent errors 

**Coherent errors**

Nature: Systematic and unitary
Cause: Miscalibration, control errors, cross-talk
Example: A rotation gate meant to be Rx(π/2) becomes Rx(π/2+ϵ)
Key feature: Errors accumulate predictably and can interfere constructively
Consequence: Can build up into large, structured deviations (not just noise—more like a consistent bias)

**Incoherent errors**

Nature: Random, non-unitary (irreversible)
Cause: Interaction with environment (noise, thermal effects)
Examples: Relaxation (T1), dephasing (T2)
Key feature: Information is lost, not just mis-rotated
Consequence: Quantum state degrades into a mixed state

Dynamical decoupling is primarily effective against **coherent errors** and low-frequency noise, because it cancels systematic phase accumulation over time. It is less effective against true decoherence, where information is irreversibly lost.


The diagram below depicts dynamical decoupling with an XX pulse sequence. The abstract circuit on the left is mapped onto a microwave pulse schedule on the top right. The bottom right depicts the same schedule, but with a sequence of two X pulses inserted during an idle period of the first qubit.

<div style="display: flex; justify-content: center;">
  <img src="DD.png" style="width: 60%; height: auto;">
</div>


Dynamical decoupling can be enabled by setting enable to True in the dynamical decoupling options. The sequence_type option can be used to pick from several different pulse sequences. The default sequence type is "XX".

In [ ]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm" # other options include "XX" and "XY4"

### Pauli twirling 

Twirling also called randomized compiling— is a technique used to reshape noise in quantum circuits into a simpler, more structured form. Instead of trying to eliminate noise entirely, the idea is to make it behave nicely.

Pauli twirling is a specific form of twirling where random Pauli operators (X,Y,Z) are inserted into the circuit. It transforms an arbitrary noise channel into a Pauli channel (i.e., noise that looks like random Pauli errors).

Coherent noise (systematic, unitary errors) can accumulate quadratically with circuit depth → things spiral quickly. Pauli (incoherent) noise accumulates linearly → much more predictable and manageable. So Pauli twirling doesn’t remove noise—it converts dangerous coherent errors into safer stochastic ones.


Pauli twirling is implemented by:
- Inserting random single-qubit Pauli gates before and after certain operations
- Choosing these Paulis carefully so that they cancel out logically, preserving the ideal computation

One circuit becomes an ensemble of randomized circuits, all implementing the same ideal unitary. When running the algorithm, you sample across many randomized versions instead of repeating a single fixed circuit

In current hardware Most errors come from two-qubit gates. So Pauli twirling is typically applied only to those gates (e.g., CNOT, ECR) as shown in examples below:


<div style="display: flex; justify-content: center;">
  <img src="PT.png" style="width: 60%; height: auto;">
</div>

Each randomized version of a gate sequence behaves differently at the noise level—but identically at the ideal level.

#### Intuition
- Without twirling: errors are structured and can align → worst-case buildup
- With twirling: errors are randomized → they average out

Pauli twirling can be enabled by setting enable_gates to True in the twirling options. Other notable options include:

- `num_randomizations`: The number of circuit instances to draw from the ensemble of twirled circuits.

- `shots_per_randomization`: The number of shots to sample from each circuit instance.

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

### Twirled Readout Error Extinction (TREX)

It is a technique for mitigating measurement (readout) errors, specifically when estimating expectation values of Pauli observables. TREX applies the same philosophy as gate twirling—but at the measurement stage. Instead of directly measuring a qubit, you randomly replace the measurement with:

- Apply an X gate
- Perform the measurement
- Flip the classical measurement result
This modified sequence is logically equivalent to a normal measurement in the absence of noise. So ideally, nothing changes.

With real hardware (where readout errors exist), this randomization has a useful effect:
- It symmetrizes the measurement errors
- The readout noise becomes diagonal in the Pauli basis
- This simplifies the error model into something much easier to correct

In practical terms:
- The readout-error transfer matrix becomes diagonal
- A diagonal matrix is trivial to invert → correction becomes straightforward and stable

Estimating the readout-error transfer matrix requires executing additional calibration circuits, which introduces a small overhead.




In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

### Zero-noise extrapolation (ZNE)

Zero-noise extrapolation (ZNE) is a technique for mitigating errors in estimating expectation values of observables. While it often improves results, it is not guaranteed to produce an unbiased result.

ZNE consists of two stages:

- Noise amplification: The original quantum circuit is executed multiple times at different noise rates.
- Extrapolation: The ideal result is estimated by extrapolating the noisy expectation value results to the zero-noise limit.

Both the noise amplification and extrapolation stages can be implemented in many different ways. Qiskit Runtime implements noise amplification by "digital gate folding," which means that two-qubit gates are replaced with equivalent sequences of the gate and its inverse. For example, replacing a unitary 
U with $U U^\dagger U$ would yield a noise amplification factor of 3. For the extrapolation, you can choose from one of several functional forms, including a linear fit or an exponential fit. The image below depicts digital gate folding on the left, and the extrapolation procedure on the right.

<div style="display: flex; justify-content: center;">
  <img src="ZNE.png" style="width: 60%; height: auto;">
</div>

The following options are notable:

- noise_factors: The noise factors to use for noise amplification.
- extrapolator: The functional form to use for the extrapolation.

In [ ]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

### Probabilistic Error Amplification (PEA)

One of the main challenges in ZNE is to accurately amplify the noise affecting the target circuit. Gate folding provides an easy way to perform this amplification, but is potentially inaccurate and might lead to incorrect results. Probabilistic error amplification provides a more accurate approach to error amplification through noise learning.

PEA is a more sophisticated technique that performs preliminary experiments to reconstruct the noise and then uses this information to perform an accurate amplification. It starts by learning the twirled noise model of each layer of entangling gates in the circuit before they are run. After the learning phase, the circuits are executed at each noise factor, where every entangling layer of the circuits is amplified by probabilistically injecting single-qubit noise proportional to the corresponding learned noise model. 

PEA consists of three stages:

- Learning: The twirled noise model of each layer of entangling gates in the circuit is learned.
- Noise amplification: The original quantum circuit is executed multiple times at different noise factors.
- Extrapolation: The ideal result is estimated by extrapolating the noisy expectation value results to the zero-noise limit.
For utility-scale experiments, PEA is often the best choice.

Because PEA is a ZNE noise amplification technique, you also need to enable ZNE by setting `resilience.zne_mitigation = True`. Other resilience.zne options can additionally be used to set extrapolators, amplification levels, and so on. PEA requires a noise model, which is automatically generated when using primitives.



In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

### Probabilistic Error Cancellation (PEC)

**Probabilistic Error Cancellation (PEC)** is an error mitigation technique used to estimate expectation values of observables.  
Unlike Zero Noise Extrapolation (ZNE), PEC provides an **unbiased estimate** of the ideal result—but at the cost of **significantly higher sampling overhead**.


PEC expresses the ideal (noise-free) operation as a linear combination of **implementable noisy operations**:

$$
O_{\text{ideal}} = \sum_i \eta_i \, O_{\text{noisy},i}
$$

- $ O_{\text{ideal}}$: Ideal observable or circuit  
- $ O_{\text{noisy},i}$: Noisy circuits that can be executed on hardware  
- $ \eta_i$: Coefficients defining the decomposition  

How PEC works?

- Instead of running a single circuit, PEC samples from a **random ensemble of noisy circuits**  
- Each circuit is chosen according to the coefficients $\eta_i$  
- Measurement results are combined to reconstruct the **ideal expectation value**

#### Quasi-probabilities and negativity

- Ideally, $ \eta_i $ would form a probability distribution  
- In practice, some $ \eta_i $ are **negative** → this forms a **quasi-probability distribution**

To handle this:
- Circuits are sampled using $ |\eta_i| $ as probabilities  
- Results are reweighted using the **sign of $ \eta_i $**  

#### Sampling overhead

The cost of PEC is determined by:

$$
\gamma = \sum_i |\eta_i| \ge 1
$$

- $ \gamma $: **Overhead factor** due to quasi-probabilities  

Key consequences:
- Number of shots required scales as **$ \gamma^2 $**  
- $ \gamma $ typically grows **exponentially with circuit depth**


In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Example usage of error mitigation 

If you want to use a noisy backend, then do:

```python
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeManilaV2 # A 5 qubit noisy backend
noisy_backend = AerSimulator.from_backend(FakeManilaV2())
```


In [27]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

# a 5 qubit GHZ state 
qc = QuantumCircuit(5)
qc.h(0)
for i in range(4):
    qc.cx(i, i+1)

observable = SparsePauliOp("Z" * 5) # expectation value of ZZZZZ, we know that this is 0 for the ideal 5 qubit GHZ state
# backend = service.least_busy(
#     operational=True, simulator=False
# )
backend= service.backend("ibm_marrakesh")
print(f"Using backend: {backend.name}") 

# Transpile our circuit specifically for this noisy backend's basis gates
pm_noisy = generate_preset_pass_manager(optimization_level=1, backend=backend)
noisy_isa_circuit = pm_noisy.run(qc) 
noisy_isa_observable = observable.apply_layout(noisy_isa_circuit.layout)

estimator = Estimator(mode=backend)
job = estimator.run([(noisy_isa_circuit, noisy_isa_observable)])
result = job.result()
print(f"Estimated expectation value on the noisy backend: {result[0].data.evs}") # expected answer =0 in the ideal case



Using backend: ibm_marrakesh
Estimated expectation value on the noisy backend: 0.02618657937806874


In [28]:
estimator_dd = Estimator(mode=backend)
 # Set options for dynamic decoupling
estimator_dd.options.dynamical_decoupling.enable = True
estimator_dd.options.dynamical_decoupling.sequence_type = "XpXm" # A common sequence
print(f"Dynamical Decoupling Enabled: {estimator_dd.options.dynamical_decoupling.enable}")
print(f"Sequence Type: {estimator_dd.options.dynamical_decoupling.sequence_type}")
 # Run the job and show the result
job_dd = estimator_dd.run([(noisy_isa_circuit, noisy_isa_observable)])
result_dd = job_dd.result()
print(f"  > Expectation value (with DD): {result_dd[0].data.evs}")

Dynamical Decoupling Enabled: True
Sequence Type: XpXm
  > Expectation value (with DD): -0.023478771277636472


In [29]:
estimator_twirl = Estimator(mode=backend)
# Set twirling options
estimator_twirl.options.twirling.enable_gates = True
estimator_twirl.options.twirling.num_randomizations = 16
print(f"Gate Twirling Enabled: {estimator_twirl.options.twirling.enable_gates}")

job_twirl = estimator_twirl.run([(noisy_isa_circuit, noisy_isa_observable)])
result_twirl = job_twirl.result()
print(f"  > Expectation value (with Twirling): {result_twirl[0].data.evs}")

Gate Twirling Enabled: True
  > Expectation value (with Twirling): -0.0031207333723425007


In [30]:
estimator_zne = Estimator(mode=backend)
# Set options for ZNE (level 0 disables presets)
estimator_zne.options.resilience_level = 0
estimator_zne.options.resilience.zne_mitigation = True
print(f"ZNE Enabled: {estimator_zne.options.resilience.zne_mitigation}")

job_zne = estimator_zne.run([(noisy_isa_circuit, noisy_isa_observable)])
result_zne = job_zne.result()
print(f"  > Expectation value (with ZNE): {result_zne[0].data.evs}")

ZNE Enabled: True
  > Expectation value (with ZNE): 0.04954759598456074


In [31]:
estimator_trex = Estimator(mode=backend)

# Set options for T-REx (level 0 disables presets)
estimator_trex.options.resilience_level = 0
estimator_trex.options.resilience.measure_mitigation = True
print(f"Measurement Mitigation (T-REx) Enabled: {estimator_trex.options.resilience.measure_mitigation}")

job_trex = estimator_trex.run([(noisy_isa_circuit, noisy_isa_observable)])
result_trex = job_trex.result()
print(f"  > Expectation value (with T-REx): {result_trex[0].data.evs}")

Measurement Mitigation (T-REx) Enabled: True
  > Expectation value (with T-REx): 0.021383401786296368


The `resilience_level` is a convenient preset that combines multiple error mitigation techniques.

In [ ]:

# --- Estimator WITH Level 1 mitigation ---
estimator_with_mitigation = Estimator(mode=backend)
estimator_with_mitigation.options.resilience_level = 1
job_with_mitigation = estimator_with_mitigation.run([(noisy_isa_circuit, noisy_isa_observable)])
result_with_mitigation = job_with_mitigation.result()

print("\n--- FINAL COMPARISON ---")
print(f"  > Expectation value (Noisy, WITH mitigation, resilience_level=1): {result_with_mitigation[0].data.evs}")


--- FINAL COMPARISON ---
  > Expectation value (Noisy, WITH mitigation, resilience_level=1): 0.011061364234922308
